In [82]:
import pandas as pd
import os
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.ensemble import RandomForestRegressor

In [83]:
# Adding some supporting functions

# Extracting temporal features
def toTime(df):
    df = df.copy()
    # Convert and extract features from measured time
    df['time'] = pd.to_datetime(df['time'], errors = 'coerce')
    df.loc[:, 'hour'] = df['time'].dt.hour
    df.loc[:, 'day'] = df['time'].dt.day
    df.loc[:, 'month'] = df['time'].dt.month
    df.loc[:, 'dayofweek'] = df['time'].dt.dayofweek
    return df

# Create a DataFrame for submission
def submit(test_ids, ylong, ylat):
    # Create a submission DataFrame
    submission = pd.DataFrame({
        'ID': test_ids,  # Pass the test ID column
        'longitude_predicted': ylong,  # Your predicted longitudes
        'latitude_predicted': ylat    # Your predicted latitudes
    })

    # Ensure the correct column order
    submission = submission[['ID', 'longitude_predicted', 'latitude_predicted']]

    # Find the next available filename by checking existing files
    i = 1
    while os.path.exists(f'jakobs_results_{i}.csv'):
        i += 1

    # Save to CSV with the next available filename
    filename = f'jakobs_results_{i}.csv'
    submission.to_csv(filename, index=False)
    print(f"Results saved to {filename}")

In [84]:
# First we read the necessary csv files. NOTE: Different delimiter for some files.
df_train = pd.read_csv('/Users/jakobrudeovstaas/Desktop/Project ML/ais_train.csv', delimiter = '|')
df_test = pd.read_csv('/Users/jakobrudeovstaas/Desktop/Project ML/ais_test.csv')
df_vessels = pd.read_csv('/Users/jakobrudeovstaas/Desktop/Project ML/vessels.csv', delimiter = '|')

# NOTE: Train is sorted by vessel ID and time for model to notice temporal trend.
df_train = df_train.sort_values(by = ['vesselId', 'time'])

In [85]:
# Extracting temporal features
df_train = toTime(df_train)
df_test = toTime(df_test)

In [86]:
# Cleaning the data. Ensuring all data is within reasonable limits and replacing outliers/erranous data with NaN
df_train['cog'] = df_train['cog'].apply(lambda x: float(x) if float(x) < 360 else pd.NA)
# NOTE: We determine that SOG above 25 knots shall be disregarded.
df_train['sog'] = df_train['sog'].apply(lambda x: float(x) if float(x) < 25 else pd.NA)
df_train['heading'] = df_train['heading'].apply(lambda x: float(x) if float(x) < 360 else pd.NA)
# NOTE: Navstat of 0 and 8 both indicate a moving vessel. We therefore combine these two
df_train['navstat'] = df_train['navstat'].replace(8, 0)

In [87]:
# Finding last known values for each unique vessel ID
last_known_locations = df_train.groupby('vesselId').agg(
    last_latitude=('latitude', 'last'),
    last_longitude=('longitude', 'last'),
    last_heading = ('heading', 'last'),
    last_sog = ('sog', 'last'),
    last_cog = ('cog', 'last')
).reset_index()

df_train = pd.merge(df_train, last_known_locations, on = 'vesselId', how = 'left')
df_test = pd.merge(df_test, last_known_locations, on = 'vesselId', how = 'left')

In [88]:
# Adding the avg SOG for each vessel ID. NOTE: mean is based on when the vessel is moving
isMoving = df_train[df_train['navstat'] == 0]
avg_sog_moving = isMoving.groupby('vesselId')['sog'].mean()
df_train['avg_sog_moving'] = df_train['vesselId'].map(avg_sog_moving)
df_test['avg_sog_moving'] = df_test['vesselId'].map(avg_sog_moving)

# Encoding labels
label_encoder = LabelEncoder()
df_train['vesselId_encoded'] = label_encoder.fit_transform(df_train['vesselId'])
df_test['vesselId_encoded'] = label_encoder.transform(df_test['vesselId'])

# Adding gross tonnage and length
df_train = pd.merge(df_train, df_vessels[['vesselId', 'GT', 'length']], on = 'vesselId', how = 'left')
df_test = pd.merge(df_test, df_vessels[['vesselId', 'yearBuilt', 'GT', 'length']], on = 'vesselId', how = 'left')

In [89]:
print(df_train.columns)

Index(['time', 'cog', 'sog', 'rot', 'heading', 'navstat', 'etaRaw', 'latitude',
       'longitude', 'vesselId', 'portId', 'hour', 'day', 'month', 'dayofweek',
       'last_latitude', 'last_longitude', 'last_heading', 'last_sog',
       'last_cog', 'avg_sog_moving', 'vesselId_encoded', 'GT', 'length'],
      dtype='object')


In [ ]:
# Implementing a RandomForestRegressor
# First defining training, test sets and target predictors.
features = ['vesselId_encoded', 'hour', 'day', 'month', 'dayofweek', 'last_latitude', 'last_longitude', 
            'last_cog', 'last_sog', 'avg_sog_moving', 'length', 'GT']
categorical = ['vesselId', 'last_navstat'] # defining categorical features for model
X_train = df_train[features].copy()
X_test = df_test[features].copy()
y_lat = df_train['latitude'].copy()
y_long = df_train['longitude'].copy()

RFRlat = RandomForestRegressor(n_estimators = 100, random_state = 42, verbose = 2)
RFRlong = RandomForestRegressor(n_estimators = 100, random_state = 42, verbose = 2)

RFRlat.fit(X_train, y_lat)
RFRlong.fit(X_train, y_long)

y_pred_lat = RFRlat.predict(X_test)
y_pred_long = RFRlong.predict(X_test)

submit(df_test['ID'], y_pred_long, y_pred_lat)

building tree 1 of 100
building tree 2 of 100
building tree 3 of 100
building tree 4 of 100
building tree 5 of 100
building tree 6 of 100
building tree 7 of 100
building tree 8 of 100
building tree 9 of 100
building tree 10 of 100
building tree 11 of 100
building tree 12 of 100
building tree 13 of 100
building tree 14 of 100
building tree 15 of 100
building tree 16 of 100
building tree 17 of 100
building tree 18 of 100
building tree 19 of 100
building tree 20 of 100
building tree 21 of 100
building tree 22 of 100
building tree 23 of 100
building tree 24 of 100
building tree 25 of 100
building tree 26 of 100
building tree 27 of 100
building tree 28 of 100
building tree 29 of 100
building tree 30 of 100
building tree 31 of 100
building tree 32 of 100
building tree 33 of 100
building tree 34 of 100
building tree 35 of 100
building tree 36 of 100
building tree 37 of 100
building tree 38 of 100
building tree 39 of 100
building tree 40 of 100


[Parallel(n_jobs=1)]: Done  40 tasks      | elapsed:  5.5min


building tree 41 of 100
building tree 42 of 100
building tree 43 of 100
building tree 44 of 100
building tree 45 of 100
building tree 46 of 100
building tree 47 of 100
building tree 48 of 100
building tree 49 of 100
building tree 50 of 100
building tree 51 of 100
building tree 52 of 100
building tree 53 of 100
building tree 54 of 100
building tree 55 of 100
building tree 56 of 100
building tree 57 of 100
building tree 58 of 100
building tree 59 of 100
building tree 60 of 100
building tree 61 of 100
building tree 62 of 100
building tree 63 of 100
building tree 64 of 100
building tree 65 of 100
building tree 66 of 100
building tree 67 of 100
building tree 68 of 100
building tree 69 of 100
building tree 70 of 100
building tree 71 of 100
building tree 72 of 100
building tree 73 of 100
building tree 74 of 100
building tree 75 of 100
building tree 76 of 100
building tree 77 of 100
building tree 78 of 100
building tree 79 of 100
building tree 80 of 100
building tree 81 of 100
building tree 82

[Parallel(n_jobs=1)]: Done  40 tasks      | elapsed:  6.1min


building tree 41 of 100
building tree 42 of 100
building tree 43 of 100
building tree 44 of 100
building tree 45 of 100
building tree 46 of 100
building tree 47 of 100
building tree 48 of 100
building tree 49 of 100
